In [4]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from typing import List

class DatasetRow(BaseModel):
    program_name: str = Field(description="Official name of the academic program")
    program_level: str = Field(description="UG, PG, Diploma, Certificate, or PhD level")
    domain: str = Field(description="Broad academic domain such as Engineering, Science, Management, Humanities, etc.")
    eligibility: str = Field(description="Eligibility criteria required for admission")
    description: str = Field(description="Short academic description of the program")
    mode: str = Field(description="Mode of the course.")
    duration: str = Field(description="The duration of the course")
    skills_learned: List[str] = Field(
        description="Key academic or professional skills gained after completing the program"
    )
    career_outcomes: List[str] = Field(
        description="Typical career paths or job roles after completing the program"
    )


model = ChatOllama(model="gemma4:e2b", temperature=0.2).with_structured_output(DatasetRow)

system_prompt="""
You are an academic data normalization and enrichment engine.
Your task is to transform raw university program information into a clean,
structured, and minimal semantic representation suitable for machine learning
and recommendation systems.
Follow ALL rules strictly.
--------------------------------
GENERAL RULES
--------------------------------
1. Use ONLY the information provided in the input.
2. Do NOT hallucinate facts, rankings, or specializations.
3. Keep wording academically neutral and precise.
4. Output MUST be valid JSON matching the required schema.
5. Do NOT include explanations, notes, or extra text outside JSON.
--------------------------------
FIELD-LEVEL TRANSFORMATION RULES
-------------------------------
program_name:
- Preserve the official academic program name exactly.
- Do not shorten or paraphrase.
program_level:
- Must be a SINGLE WORD chosen from:
  ["PreUG", "UG", "PG", "PhD"]
domain:
- Must be a SINGLE WORD academic domain keyword chosen from: ["Engineering", "Science", "Management", "Commerce", "Humanities"]  
- Choose the most appropriate broad discipline.
eligibility:
- Rewrite into a **simple, clear, single-sentence requirement**.
- Keep the original explicit requirements while shortening it
example = "Bachelor's Degree in [subject] with at least %age marks (%age marks for reserved categories)"
description:
- Write a **2–3 sentence academic summary** of what the program studies.
- Focus on knowledge areas, learning scope, and discipline.
- Do NOT mention university names, rankings, or admissions process.
mode:
- preserve the original mode of study mentioned and return as string.
duration:
- preserve the original duration of the course and return as string.
skills_learned:
- Provide **3 to 6 concise skill phrases**.
- Each item must be SHORT (2–4 words).
- Skills should be abstract not explicitly related to the field.
- Example:
  ["Data analysis", "Laboratory techniques", "Research methodology"]
career_outcomes:
- Provide **3 to 6 clear and descriptive career paths**.
- Each item should be a **readable professional role**.
- Example:
  ["Research scientist", "Clinical laboratory specialist"]
"""

In [ ]:
import pandas as pd
import time
from langchain_core.messages import HumanMessage, SystemMessage
df = pd.read_excel("courses_data.xlsx")
results = []

for idx, row in df.iterrows():
    try:
        info = f"""
        Program Name: {row.get('Program Name', '')}
        Program Level: {row.get('Program Level', '')}
        Eligibility: {row.get('Eligibility', '')}
        mode: {row.get("Mode")}
        Duration: {row.get("Duration")}
    """
        conversation = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=info)
        ]
        response: DatasetRow = model.invoke(conversation)

        results.append(response.model_dump())
        print(f"{idx+1} row processed")
        time.sleep(0.5)
    except Exception as e:
        print(f"Error at row: {idx+1}")
            
    

buffer_df = pd.DataFrame(results)

Processed batch 1
Processed batch 1
Processed batch 1
Processed batch 1
Processed batch 1
Processed batch 1
Processed batch 1
Processed batch 1
Processed batch 1
Processed batch 1
Processed batch 11
Processed batch 11
Processed batch 11
Processed batch 11
Processed batch 11
Processed batch 11
Processed batch 11
Processed batch 11
Processed batch 11
Processed batch 11
Processed batch 21
Processed batch 21
Processed batch 21
Processed batch 21
Processed batch 21
Processed batch 21
Processed batch 21
Processed batch 21
Processed batch 21


KeyboardInterrupt: 

In [ ]:
buffer_df.to_csv("buffer_data.csv", index=False)

In [2]:
system_prompt = """
You are a deterministic academic eligibility extraction engine.

Your task is to extract ONLY the minimum eligibility requirements from university course eligibility data.

OUTPUT REQUIREMENTS:
- Return ONLY valid JSON matching the exact Pydantic schema provided
- No explanations, no commentary, no markdown
- If information is missing, use null for optional fields

================================
FIELD 1: min_degree_level
================================

Extract the MINIMUM educational qualification required.

Map to EXACTLY one of these values:
- "PreUG"  → For 10+2, Higher Secondary, Senior Secondary, Intermediate, or equivalent
- "UG"     → For Bachelor's degree, Graduation, Undergraduate degree
- "PG"     → For Master's degree, Postgraduate degree
- "PhD"    → For Doctoral degree, Ph.D.

MAPPING RULES:
- "10+2" or "12th" or "Higher Secondary" → PreUG
- "Senior Secondary Certificate" → PreUG
- "Intermediate" or "10+2 level" → PreUG
- "Bachelor" or "Bachelor's" or "B.A." or "B.Sc." or "B.Tech" or "B.E." → UG
- "Graduation" or "Graduate degree" → UG
- "Master" or "Master's" or "M.A." or "M.Sc." or "M.Tech" or "MBA" → PG
- "Postgraduate" or "Post Graduate" → PG
- "Ph.D." or "Doctorate" or "Doctoral" → PhD

If text says "Master's OR Bachelor's with experience" → return "UG" (the lower requirement)
If text says "any degree" without specifying level → return "PreUG" (most permissive)
If completely unclear or missing → return null

================================
FIELD 2: min_marks_general
================================

Extract the MINIMUM percentage marks required for GENERAL category students.

EXTRACTION RULES:
- Look for phrases like "50%", "minimum 50% marks", "at least 50 percent"
- Return as a float: 50.0, 60.0, 55.0, etc.
- Extract ONLY the general category requirement
- IGNORE SC/ST/OBC/Reserved category marks (we extract those separately)

COMMON PATTERNS:
- "50% marks (45% for SC/ST)" → return 50.0 only
- "Minimum 60% in graduation" → return 60.0
- "At least 55 percent" → return 55.0
- "50% or equivalent CGPA" → return 50.0
- "Pass marks" or "No minimum percentage" → return null

If marks not mentioned at all → return null
If only says "pass" without percentage → return null

================================
FIELD 3: min_marks_reserved
================================

Extract the MINIMUM percentage marks for RESERVED category (SC/ST/OBC/PwD).

EXTRACTION RULES:
- Look for reserved/SC/ST/OBC/Disabled/PwD category marks
- Return as float: 45.0, 47.5, 40.0, etc.
- Only extract if EXPLICITLY mentioned

COMMON PATTERNS:
- "50% marks (45% for SC/ST)" → return 45.0
- "55% (50% for reserved categories)" → return 50.0
- "60% or 55% for SC/ST/OBC" → return 55.0
- If no reserved category mentioned → return null

================================
EDGE CASES & SPECIAL HANDLING
================================

1. Multiple conditions (take the minimum):
   - "Bachelor's with 50% OR Master's with 45%" → min_degree_level: "UG", marks: 45.0

2. Subject-specific marks:
   - "50% aggregate with 60% in Mathematics" → return 50.0 (aggregate is the requirement)

3. CGPA mentioned:
   - "5.5 CGPA or 55%" → return 55.0 (use percentage when both given)
   - "6.0 CGPA" → return 60.0 (approximate: CGPA*10)

4. Grade-based:
   - "First Division" → return 60.0 (standard conversion)
   - "Second Division" → return 50.0
   - If only grade mentioned without % → return null

5. Entrance exam instead of marks:
   - "CUET score required" with no marks → return null for marks

6. Experience as alternative:
   - "UG with 50% OR Diploma with 2 years experience" → return "UG" and 50.0

================================
EXAMPLES
================================

Input: "Bachelor's Degree in any discipline with minimum 50% marks (45% for SC/ST candidates)"
Output: {
  "min_degree_level": "UG",
  "min_marks_general": 50.0,
  "min_marks_reserved": 45.0
}

Input: "10+2 or equivalent with 50% marks"
Output: {
  "min_degree_level": "PreUG",
  "min_marks_general": 50.0,
  "min_marks_reserved": null
}

Input: "Master's degree in Science with 55% (50% for reserved)"
Output: {
  "min_degree_level": "PG",
  "min_marks_general": 55.0,
  "min_marks_reserved": 50.0
}

Input: "Graduation in any stream, no minimum percentage"
Output: {
  "min_degree_level": "UG",
  "min_marks_general": null,
  "min_marks_reserved": null
}

Input: "Ph.D. with First Division"
Output: {
  "min_degree_level": "PhD",
  "min_marks_general": 60.0,
  "min_marks_reserved": null
}

Input: "Any qualification, admission through entrance test"
Output: {
  "min_degree_level": "PreUG",
  "min_marks_general": null,
  "min_marks_reserved": null
}

================================
CRITICAL REMINDERS
================================

1. ALWAYS return valid JSON matching the schema
2. Use EXACT degree level strings: PreUG, UG, PG, or PhD
3. Return float for marks: 50.0, 60.0, 55.5 (NOT integers, NOT strings)
4. When in doubt about marks → return null (it's better than guessing)
5. Extract minimum requirements (most permissive interpretation)
6. DO NOT hallucinate information not present in input
7. DO NOT include explanations in output

Now extract eligibility from the following course data:
"""

## Iterative restructure pipeline

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
import pandas as pd
import time

class EligibilityStruct(BaseModel):
    min_degree_level: Optional[str] = Field(
        default=None,
        description="One of: PreUG, UG, PG, PhD"
    )
    min_marks_general: Optional[float] = Field(
        default=None,
        description="Minimum percentage for general category",
        ge=0.0,
        le=100.0
    )
    min_marks_reserved: Optional[float] = Field(
        default=None,
        description="Minimum percentage for reserved category (SC/ST/OBC/PwD)",
        ge=0.0,
        le=100.0
    )

class DatasetRow(BaseModel):
    program_name: str = Field(description="Official name of the academic program")
    program_level: str = Field(description="UG, PG, Diploma, Certificate, or PhD level")
    domain: str = Field(description="Broad academic domain such as Engineering, Science, Management, Humanities, etc.")
    eligibility: str = Field(description="Eligibility criteria required for admission")
    description: str = Field(description="Short academic description of the program")
    skills_learned: List[str] = Field(
        description="Key academic or professional skills gained after completing the program"
    )
    career_outcomes: List[str] = Field(
        description="Typical career paths or job roles after completing the program"
    )
    eligibility_struct: EligibilityStruct = Field(description="Eligibiliy in the structured format")

model = ChatOllama(model="qwen2.5:7b", tempurature=0.2)

df = pd.read_csv("final_data.csv")
results = []
print(f"Starting iterative Data Modeling")

batch_size = 10
for i in range(0, len(df), batch_size):
    
    batch = df.iloc[i:i+batch_size]

    for idx, row in batch.iterrows():
        eligibility = row["eligibility"]
        conversation = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=eligibility)
        ]
        response = model.invoke(conversation).model_dump()  
        results.append(response)
        time.sleep(0.5)
    
    print(f"Processed batch {i+1}")

print(f"Created {len(results)} records.")

Starting iterative Data Modeling
Processed batch 1
Processed batch 11
Processed batch 21
Processed batch 31
Processed batch 41
Processed batch 51
Processed batch 61
Processed batch 71
Processed batch 81
Processed batch 91
Processed batch 101
Processed batch 111
Created 114 records.


In [6]:
# Saving records to new field
df["eligibility_struct"] = results
df.to_csv("final_data2.csv")